# Работа с базами данных

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Работа с базами данных"
* https://sqliteonline.com/
* https://docs.python.org/3/library/sqlite3.html
* https://www.sqlitetutorial.net/sqlite-create-table/
* https://docs.python.org/3/library/pickle.html
* https://www.geeksforgeeks.org/sql-join-set-1-inner-left-right-and-full-joins/
* https://www.datacamp.com/community/tutorials/group-by-having-clause-sql

## Задачи для совместного разбора

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
students = pd.DataFrame(
    [
        ("Сотников Евгений Янович", 1),
        ("Степанова Виктория Константиновна", 1),
        ("Горелова Вероника Яновна", 2),
        ("Гришин Иван Романович", 3),
    ],
    columns=["name", "group_id"],
)


groups = list(zip([1, 2, 3], ["ПМ20-1", "ПМ20-2", "ПМ20-3"]))

In [ ]:
groups, students

([(1, 'ПМ20-1'), (2, 'ПМ20-2'), (3, 'ПМ20-3')],
                                 name  group_id
 0            Сотников Евгений Янович         1
 1  Степанова Виктория Константиновна         1
 2           Горелова Вероника Яновна         2
 3              Гришин Иван Романович         3)

1. Создайте БД sqlite3 и таблицы Student и StudentGroup в ней.

In [ ]:
con = sqlite3.connect('demo2.db')

In [ ]:
cur = con.cursor()

In [ ]:
sql = '''
CREATE TABLE StudentGroup(
    id INT PRIMARY KEY,
    name VARCHAR
);

CREATE TABLE Student(
    name VARCHAR PRIMARY KEY,
    group_id INT,
    FOREIGN KEY (group_id) REFERENCES StudentGroup(id)
);
'''

In [ ]:
cur.executescript(sql)
con.commit() #подтверждение, что все, что выше надо сохранить в базе

2. Заполните созданные таблицы данными

In [ ]:
#sql = '''
#INSERT INTO StudentGroup(id, name)
#VALUES
#    (1, 'ПМ21-1'),
#    (2, 'ПМ21-2'),
#    (3, 'ПМ21-3')'''

In [ ]:
sql = '''
INSERT INTO StudentGroup(id, name) VALUES (?, ?)
'''
#for group in groups:
#    cut.execute(sql, group)
# ИЛИ более удобный вариант:
cur.executemany(sql, groups)
con.commit() #чтобы быть уверенным, что все добавилось

In [ ]:
students.to_sql('Student', con, if_exists='append', index=False)
con.commit()

3. Выведите на экран фамилии студентов и номера их групп.

In [ ]:
r = cur.execute('SELECT * FROM StudentGroup;')
#r.fetchone()
#r.fetchmany(2)
r.fetchall()

[(1, 'ПМ20-1'), (2, 'ПМ20-2'), (3, 'ПМ20-3')]

In [ ]:
pd.read_sql_query('SELECT * FROM Student;', con)

,name,group_id
0,Сотников Евгений Янович,1
1,Степанова Виктория Константиновна,1
2,Горелова Вероника Яновна,2
3,Гришин Иван Романович,3


In [ ]:
cur.close()
con.close()

## Лабораторная работа 3

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy` и `pandas`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy` или структур `pandas` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

### Создание и заполнение базы данных

<p class="task" id="1"></p>

1\. Создайте файл БД sqlite3 согласно рисунку ниже, на котором определен набор таблиц и связей между ними. Обратите внимание, что поля, выделенные полужирным шрифтом, обозначают первичный ключ таблицы.

Для решения задания напишите скрипт на языке SQL и исполните его при помощи метода `executescript` объекта-курсора.

![image-2.png](attachment:image-2.png)

In [ ]:
con = sqlite3.connect('DataBase3.db')

In [ ]:
cur = con.cursor()

In [ ]:
sql = '''
CREATE TABLE Recipe(
  id INT PRIMARY KEY,
  name VARCHAR,
  minutes INT,
  submitted VARCHAR,
  description TEXT,
  n_ingredients INT
);

CREATE TABLE Tag(
  tag VARCHAR,
  recipe_id INT,
  PRIMARY KEY (tag, recipe_id)
  FOREIGN KEY (recipe_id) REFERENCES Recipe (id)
);

CREATE TABLE Review(
  id INT PRIMARY KEY,
  user_id INT,
  recipe_id INT,
  date VARCHAR,
  rating INT,
  review TEXT,
  FOREIGN KEY (recipe_id) REFERENCES Recipe (id)
)
'''

In [ ]:
cur.executescript(sql)
con.commit()

In [ ]:
cur.close()
con.close()

<p class="task" id="2"></p>

2\. Загрузите данные из файла `recipes_sample.csv` в таблицу `Recipe`. При выполнении задания воспользуйтесь методом `executemany` объекта-курсора.

In [ ]:
con = sqlite3.connect('DataBase3.db')
cur = con.cursor()

In [ ]:
recipes = pd.read_csv('/Users/ivanlopatkin/Downloads/recipes_sample.csv')
recipes.head()

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN


In [ ]:
lst = [tuple(row) for row in recipes.drop(['contributor_id', 'n_steps'], axis=1).values]

In [ ]:
sql = '''
INSERT INTO Recipe (name, id, minutes, submitted, description, n_ingredients)
VALUES
    (?, ?, ?, ?, ?, ?)
'''

In [ ]:
cur.executemany(sql, lst)
con.commit()

In [ ]:
r = cur.execute('SELECT * FROM Recipe;')
r.fetchmany(2)

[(44123,
  'george s at the cove  black bean soup',
  90,
  '2002-10-25',
  "an original recipe created by chef scott meskan, george's at the cove. we enjoyed this when we visited this restaurant in la jolla, california. this recipe is requested so often, they have it printed and ready at the hostess stand. it's unbeatable at the restaurant, but i do a pretty good job at home, too, if i do say so myself!",
  18),
 (67664,
  'healthy for them  yogurt popsicles',
  10,
  '2003-07-26',
  'my children and their friends ask for my homemade popsicles morning, noon and night. i never turn them down; who am i to tell them that they are good for them! for variety i substitute different flavours of frozen juice - grape, fruit punch, tropical etc.',
  None)]

<p class="task" id="3"></p>

3\. Загрузите данные из файла `reviews_sample.csv` в таблицу `Review`. При выполнении задания воспользуйтесь методом `pd.DataFrame.to_sql`.

In [ ]:
reviews = pd.read_csv('/Users/ivanlopatkin/Downloads/reviews_sample.csv')
reviews = reviews.rename(columns={'Unnamed: 0':'id'})

In [ ]:
reviews.to_sql('Review', con, if_exists='append', index=False)
con.commit()

In [ ]:
pd.read_sql_query("SELECT * FROM Review;", con)

,id,user_id,recipe_id,date,rating,review
0,370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
1,624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
2,187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
3,706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
4,312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...
...,...,...,...,...,...,...
126691,1013457,1270706,335534,2009-05-17,4,This recipe was great! I made it last night. I...
126692,158736,2282344,8701,2012-06-03,0,This recipe is outstanding. I followed the rec...
126693,1059834,689540,222001,2008-04-08,5,"Well, we were not a crowd but it was a fabulou..."
126694,453285,2000242659,354979,2015-06-02,5,I have been a steak eater and dedicated BBQ gr...


<p class="task" id="4"></p>

4\. Загрузите данные из файла `tags_sample.pickle` в таблицу `Tag`. При выполнении задания воспользуйтесь методом `executemany` объекта-курсора или методом `pd.DataFrame.to_sql`.

Для считывания файла с данными воспользуйтесь пакетом `pickle`. Обратите внимание, что перед добавлением записей в базу данные нужно привести к соответствующему таблице в БД виду (в каждой строчке столбца tag должен находиться один тэг).

In [ ]:
import pickle
from scipy import concatenate
import numpy as np

In [ ]:
with open('/Users/ivanlopatkin/Desktop/tags_sample.pickle', 'rb') as f:
    tags = pickle.load(f)
tags[0]

{'id': 48,
 'tag': {'4-hours-or-less',
  'american',
  'course',
  'cuisine',
  'desserts',
  'dietary',
  'eggs-dairy',
  'equipment',
  'main-ingredient',
  'north-american',
  'oven',
  'pies',
  'pies-and-tarts',
  'preparation',
  'time-to-make',
  'weeknight'}}

In [ ]:
len(tags)

29984

1 способ

In [ ]:
tags_sample = pd.DataFrame(tags)
tags_sample['tag'] = tags_sample['tag'].apply(lambda x: str(list(x)).replace('[','').replace(']',''))

In [ ]:
tags_sample.head()

,id,tag
0,48,"'equipment', 'dietary', 'time-to-make', 'ameri..."
1,55,"'healthy', 'easy', 'southwestern-united-states..."
2,66,"'healthy', 'easy', 'lactose', 'southern-united..."
3,91,"'course', 'pasta-rice-and-grains', '4-hours-or..."
4,94,"'occasion', 'berries', 'finger-food', 'equipme..."


In [ ]:
tags_sample['tag'] = tags_sample['tag'].str.split(', ')
tags_sample = tags_sample.explode('tag')

In [ ]:
tags_sample = tags_sample.reset_index(drop=True)

In [ ]:
tags_sample.columns = ['recipe_id', 'tag']

In [ ]:
tags_sample.to_sql('Tag', con, if_exists='append', index = False)

2 способ

In [ ]:
df = pd.DataFrame({'tag': [], 'recipe_id': []})

In [ ]:
for i in range(len(tags)):
    tag_list = list(tags[i]['tag'])
    df = df.append(pd.DataFrame({'tag': tag_list, 'recipe_id': [tags[i]['id']] * len(tag_list)}))

In [ ]:
df['recipe_id'] = df['recipe_id'].astype(int)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.tail()

,tag,recipe_id
533469,4-hours-or-less,536747
533470,number-of-servings,536747
533471,for-large-groups,536747
533472,time-to-make,536747
533473,desserts,536747


In [ ]:
df.to_sql('Tag', con, if_exists='append', index=False)
con.commit()

---
### Получение данных из базы

<p class="task" id="5"></p>

5\. Напишите и выполните запрос на языке SQL, который считает кол-во рецептов, опубликованных в 2010 году и имеющих длину не менее 15 минут. Для выполнения запроса используйте метод `execute` объекта-курсора. Выведите искомое количество на экран.

In [ ]:
sql = '''
SELECT COUNT(*) FROM Recipe
WHERE submitted LIKE '2010%' AND minutes >= 15
'''

In [ ]:
cur.execute(sql)
con.commit()

In [ ]:
cur.fetchone()[0]

1319

<p class="task" id="6"></p>

6\. Напишите и выполните запрос на языке SQL, который возращает id рецептов, не имеющих ни одного отзыва отзывов с рейтингом, меньше 4. Для выполнения запроса используйте функцию `pd.read_sql_query`. Выведите полученный результат на экран.

In [ ]:
# Не совсем правильное решение, так как некоторых данных не хватает
sql = '''
SELECT recipe_id FROM Review
GROUP BY recipe_id
HAVING MIN(rating) >= 4;
    '''

In [ ]:
pd.read_sql_query(sql, con)

,recipe_id
0,55
1,66
2,91
3,94
4,128
...,...
20461,536360
20462,536473
20463,536547
20464,536728


In [ ]:
# Правильное
sql = '''
SELECT id FROM Recipe WHERE id NOT IN (SELECT recipe_id FROM Review WHERE rating < 4)
'''

In [ ]:
pd.read_sql_query(sql, con)

,id
0,55
1,66
2,91
3,94
4,128
...,...
22361,536360
22362,536473
22363,536547
22364,536728


7\. Используя механизмы группировки и объединения, которые предоставляет SQL, выведите на экран названия и количество тегов 5 рецептов, которые имеют наибольшее количество тэгов. При выполнении задания воспользуйтесь методом `execute` объекта-курсора. Измерьте время выполнения работы вашего кода.

Вся необходимая логика (группировки, объединения, выбор топ-5 строк) должна быть реализована на SQL, а не в виде кода на Python.

In [ ]:
sql = '''
SELECT name, COUNT(*) AS count_tags
FROM Recipe
INNER JOIN Tag
    ON Recipe.id = Tag.recipe_id
GROUP BY Recipe.id
ORDER BY count_tags DESC
LIMIT 5;
'''

In [ ]:
cur.execute(sql)
con.commit()

In [ ]:
cur.fetchall()

[('watermelon basket fruit salad', 58),
 ('creamsicle freeze', 56),
 ('easy poached salmon with dill', 53),
 ('curried crab asparagus cheesy tofu dip', 53),
 ('fusion ketchup', 52)]

<p class="task" id="8"></p>

8\. Запросите у пользователя id рецепта и верните информацию об этом рецепте. Если рецепт отсутствует, выведите соответствующее сообщение. Для подстановки значения id необходимо воспользоваться специальным синтаксисом, которые предоставляет `sqlite` для этих целей. Продемонстрируйте работоспособность вашего решения.

In [ ]:
recipe_id = input('Введите id: ')

sql = '''
SELECT *
FROM Recipe
WHERE id=?
'''
cur.execute(sql,(recipe_id,))

res = cur.fetchone()
if res is None:
    print('Рецепт отсутствует')
else:
    print(res)

Введите id:  23


Рецепт отсутствует


In [ ]:
cur.close()
con.close()